In [ ]:
import math
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from tgb.nodeproppred.dataset import NodePropPredDataset


ROOT = Path(__file__).resolve().parent
CIK_PATH = ROOT / "cik_features.csv"
CUSIP_PATH = ROOT / "cusip_features.csv"
CIK_OUT_PATH = ROOT / "cik_features_normalized.csv"
CUSIP_OUT_PATH = ROOT / "cusip_features_normalized.csv"
NODE_FEAT_OUT_PATH = ROOT / "node_feat_normalized.npy"

# Columns to log1p before standardisation
CIK_LOG_COLS = {"AUM", "num_stocks"}
CUSIP_LOG_COLS = {
    "num_holders",
    "total_institutional_ownership",
    "total_shares_held",
    "avg_holding_per_manager",
    "volatility_of_holdings",
}

# Columns to skip when z-scoring
CIK_EXCLUDE_COLS = {"CIK", "source_file"}
CUSIP_EXCLUDE_COLS = {"CUSIP", "source_file", "sector"}


def _parse_source_file(source_file: str) -> pd.Timestamp:
    year = int(source_file[:4])
    quarter = source_file[4:6].lower()
    quarter_map = {"q1": "03-31", "q2": "06-30", "q3": "09-30", "q4": "12-31"}
    return pd.to_datetime(f"{year}-{quarter_map[quarter]}")


def _log1p_columns(df: pd.DataFrame, cols: Iterable[str]) -> None:
    for col in cols:
        if col in df.columns:
            df[col] = np.log1p(df[col].clip(lower=0))


def _zscore(df: pd.DataFrame, exclude: Iterable[str]) -> pd.DataFrame:
    result = df.copy()
    value_cols = [c for c in df.columns if c not in exclude]
    for col in value_cols:
        std = df[col].std(ddof=0)
        if math.isclose(std, 0.0):
            continue
        result[col] = (df[col] - df[col].mean()) / std
    return result


def _normalize_table(csv_path: Path, log_cols, exclude_cols) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if df["source_file"].dtype == object:
        unique_files = sorted(df["source_file"].unique(), key=_parse_source_file)
        dataset = NodePropPredDataset(name="tgbn-13f", root="datasets")
        mapping = dict(zip(unique_files, dataset.label_ts))
        df["source_file"] = df["source_file"].map(mapping)
    _log1p_columns(df, log_cols)
    return _zscore(df, exclude_cols)


def _build_temporal_tensor(
    dataset: NodePropPredDataset,
    cik_df: pd.DataFrame,
    cusip_df: pd.DataFrame,
) -> np.ndarray:
    node_ids = dataset.node_ids
    label_ts = list(dataset.label_ts)
    cik_cols = [c for c in cik_df.columns if c not in ("CIK", "source_file")]
    cusip_cols = [c for c in cusip_df.columns if c not in ("CUSIP", "source_file")]
    feat_dim = len(cik_cols) + len(cusip_cols)
    tensor = np.zeros((len(label_ts), len(node_ids), feat_dim), dtype=np.float32)

    cik_lookup = cik_df.set_index(["CIK", "source_file"])
    cusip_lookup = cusip_df.set_index(["CUSIP", "source_file"])
    ts_index = {ts: i for i, ts in enumerate(label_ts)}

    for node_name, node_idx in node_ids.items():
        for ts in label_ts:
            slice_idx = ts_index[ts]
            if (node_name, ts) in cik_lookup.index:
                tensor[slice_idx, node_idx, len(cusip_cols):] = cik_lookup.loc[(node_name, ts), cik_cols].values
            elif (node_name, ts) in cusip_lookup.index:
                tensor[slice_idx, node_idx, : len(cusip_cols)] = cusip_lookup.loc[(node_name, ts), cusip_cols].values
    return tensor

dataset = NodePropPredDataset(name="tgbn-13f", root="datasets")
cik_norm = _normalize_table(CIK_PATH, CIK_LOG_COLS, CIK_EXCLUDE_COLS)
cusip_norm = _normalize_table(CUSIP_PATH, CUSIP_LOG_COLS, CUSIP_EXCLUDE_COLS)

cik_norm.to_csv(CIK_OUT_PATH, index=False)
cusip_norm.to_csv(CUSIP_OUT_PATH, index=False)

node_tensor = _build_temporal_tensor(dataset, cik_norm, cusip_norm)
np.save(NODE_FEAT_OUT_PATH, node_tensor)
print(f"Saved normalized CIK features -> {CIK_OUT_PATH}")
print(f"Saved normalized CUSIP features -> {CUSIP_OUT_PATH}")
print(f"Saved temporal node features -> {NODE_FEAT_OUT_PATH}")